# EXP21D-v6 — Fair Control 1C: Frozen Random GraphSAGE
Random-init GraphSAGE `[15, 10]` is frozen at zero training steps. The same checkpoint is reused for Uniform, Topology, and Importance sampling; only downstream XGBoost sees labels. All samplers run sequentially inside this notebook.


In [ ]:
%%writefile requirements.txt
numpy>=1.26
pandas>=2.1
scikit-learn>=1.4
matplotlib>=3.8
PyYAML>=6.0
torch>=2.2
torch-geometric>=2.5
pyg_lib>=0.4
xgboost>=2.0


In [ ]:
%%writefile config.kaggle.yaml
experiment:
  id: "EXP21D-v6-FairCtrl-1C"
  seed: 42
  encoder_seed: 42
  evaluation_seed: 1042
data:
  transactions: "/kaggle/input/credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  start_year: 2016
  max_rows: null
time_window: {reference_date: "2018-01-01", train_days: 300, val_days: 30, gap_days: 8, test_days: 60}
features: {mode: "clean", use_cyclical_time: true, use_amount_log: true, input_dim_expected: 10}
sampling:
  methods: ["uniform", "topology", "importance"]
  undersampling_ratio: 1.0
  smoothing_alpha: 0.1
  weight_attr: "sampling_weight"
model: {hidden_channels: 128, out_channels: 1, num_layers: 2, aggr: "mean", dropout: 0.2}
training: {batch_size: 1024, num_neighbors: [15, 10]}
classifier:
  n_estimators: 100
  max_depth: 4
  learning_rate: 0.05
  min_child_weight: 10
  subsample: 0.8
  colsample_bytree: 0.8
  scale_pos_weight_strategy: "sqrt"
  tree_method: "hist"
  n_jobs: -1
threshold: {min_precision: 0.15}
reproducibility: {deterministic: true, warn_only: true, compute_dataset_sha256: false}
output:
  model_dir: "/kaggle/working/model/exp21d_v6_fairctrl_1c"
  result_dir: "/kaggle/working/result/exp21d_v6_fairctrl_1c"
  save_feature_matrices: true
  snapshot_source: true
smoke: {max_rows_per_split: 20000}


In [ ]:
%%writefile config.py
from pathlib import Path

import yaml


def load_config(config_path: str = "config.reference.yaml") -> dict:
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Berkas konfigurasi tidak ditemukan: {path}")
    with path.open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    if config.get("training", {}).get("num_neighbors") != [15, 10]:
        raise ValueError("EXP21D-v6-1C mengunci num_neighbors=[15, 10]")
    if float(config.get("sampling", {}).get("undersampling_ratio", 1.0)) != 1.0:
        raise ValueError("EXP21D-v6-1C melarang graph undersampling; encoder harus label-independent")
    if config.get("sampling", {}).get("weight_attr") != "sampling_weight":
        raise ValueError("EXP21D-v6-1C mewajibkan weight_attr='sampling_weight'")
    if int(config.get("model", {}).get("hidden_channels", 0)) != 128:
        raise ValueError("EXP21D-v6-1C mengunci dimensi embedding=128")
    return config


In [ ]:
%%writefile artifacts.py
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def write_json(path: str | Path, value: Any) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")
    temporary.replace(target)


class RunLogger:
    """Append-only event history; a crashed run remains auditable."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, event: str, **payload: Any) -> None:
        record = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "event": event, **payload}
        with self.path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def build_artifact_index(run_dir: str | Path) -> dict[str, Any]:
    root = Path(run_dir)
    entries = []
    for item in sorted(root.rglob("*")):
        if item.is_file() and item.name != "artifact_index.json":
            entries.append({"path": item.relative_to(root).as_posix(), "size_bytes": item.stat().st_size})
    result = {"run_dir": str(root.resolve()), "file_count": len(entries), "files": entries}
    write_json(root / "artifact_index.json", result)
    return result



In [ ]:
%%writefile reproducibility.py
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any

import numpy as np


def stable_hash(value: Any, length: int = 12) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:length]


def seed_everything(seed: int, deterministic: bool = True, warn_only: bool = True) -> None:
    import torch

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=warn_only)


def file_fingerprint(path: str | Path, with_sha256: bool = False) -> dict[str, Any]:
    source = Path(path).resolve()
    stat = source.stat()
    result: dict[str, Any] = {"path": str(source), "size_bytes": stat.st_size, "modified_time_ns": stat.st_mtime_ns}
    if with_sha256:
        digest = hashlib.sha256()
        with source.open("rb") as handle:
            while block := handle.read(8 * 1024 * 1024):
                digest.update(block)
        result["sha256"] = digest.hexdigest()
    return result


def git_revision(path: str | Path) -> dict[str, Any]:
    try:
        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=path, text=True, stderr=subprocess.DEVNULL).strip()
        dirty = bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=path, text=True).strip())
        return {"commit": commit, "dirty": dirty}
    except (OSError, subprocess.CalledProcessError):
        return {"commit": None, "dirty": None}


def environment_manifest() -> dict[str, Any]:
    import pandas
    import sklearn
    import torch
    import xgboost

    return {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pandas.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__,
        "xgboost": xgboost.__version__,
        "cuda_runtime": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }


def snapshot_source(source_dir: str | Path, destination: str | Path) -> None:
    source, target = Path(source_dir), Path(destination)
    target.mkdir(parents=True, exist_ok=True)
    for item in source.iterdir():
        if item.is_file() and item.suffix.lower() in {".py", ".yaml", ".yml", ".md", ".txt"}:
            shutil.copy2(item, target / item.name)



In [ ]:
%%writefile data_preprocessing.py
from __future__ import annotations

from datetime import timedelta

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def load_and_split_raw_data(
    file_path: str,
    start_year: int = 2016,
    nrows: int | None = None,
    reference_date: str = "2018-01-01",
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    """Load IBM transactions and create the locked 300/30/8/60 temporal split."""
    chunks: list[pd.DataFrame] = []
    collected = 0
    for chunk in pd.read_csv(file_path, chunksize=1_000_000):
        if start_year is not None:
            chunk = chunk[chunk["Year"] >= start_year]
        if nrows is not None:
            chunk = chunk.iloc[: max(0, nrows - collected)]
        if len(chunk):
            chunks.append(chunk)
            collected += len(chunk)
        if nrows is not None and collected >= nrows:
            break
    if not chunks:
        raise ValueError("Tidak ada transaksi setelah filter data diterapkan")

    frame = pd.concat(chunks, ignore_index=True)
    frame["transaction_id"] = np.arange(len(frame), dtype=np.int64)
    frame["Time"] = frame["Time"].fillna("00:00")
    frame["datetime"] = pd.to_datetime(
        frame["Year"].astype(str)
        + "-"
        + frame["Month"].astype(str).str.zfill(2)
        + "-"
        + frame["Day"].astype(str).str.zfill(2)
        + " "
        + frame["Time"]
    )
    frame = frame.sort_values(["datetime", "transaction_id"]).reset_index(drop=True)
    frame["Amount"] = (
        frame["Amount"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
    )
    frame["Is Fraud?"] = frame["Is Fraud?"].map({"Yes": 1, "No": 0})
    if frame["Is Fraud?"].isna().any():
        raise ValueError("Label Is Fraud? memuat nilai selain Yes/No")

    start = pd.Timestamp(reference_date)
    val_start = start + timedelta(days=train_days)
    gap_start = val_start + timedelta(days=val_days)
    test_start = gap_start + timedelta(days=gap_days)
    test_end = test_start + timedelta(days=test_days)

    def take(lower: pd.Timestamp, upper: pd.Timestamp) -> pd.DataFrame:
        return frame[(frame["datetime"] >= lower) & (frame["datetime"] < upper)].copy().reset_index(drop=True)

    train = take(start, val_start)
    validation = take(val_start, gap_start)
    gap = take(gap_start, test_start)
    test = take(test_start, test_end)
    splits = {"train": train, "validation": validation, "gap": gap, "test": test}
    if any(len(value) == 0 for value in splits.values()):
        raise ValueError(f"Ada split temporal kosong: { {key: len(value) for key, value in splits.items()} }")

    statistics = {
        "protocol": "SHINE FRAUD II 300d/30d/8d/60d",
        "boundaries": {
            "train": [str(start), str(val_start)],
            "validation": [str(val_start), str(gap_start)],
            "embargo_gap": [str(gap_start), str(test_start)],
            "test": [str(test_start), str(test_end)],
        },
        "splits": {
            name: {
                "rows": int(len(value)),
                "fraud": int(value["Is Fraud?"].sum()) if name != "test" else None,
                "fraud_rate": float(value["Is Fraud?"].mean()) if name != "test" else None,
                "first_timestamp": str(value["datetime"].min()),
                "last_timestamp": str(value["datetime"].max()),
            }
            for name, value in splits.items()
        },
    }
    return train, validation, test, gap, statistics


def apply_strict_anti_leakage_preprocessing(
    train: pd.DataFrame,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    gap: pd.DataFrame,
    feature_mode: str = "clean",
    use_cyclical_time: bool = True,
    use_amount_log: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str], dict]:
    """Fit every learned transform on train only and map unseen categories to zero."""
    if feature_mode not in {"clean", "clean_basic"}:
        raise ValueError("feature_mode harus clean atau clean_basic")
    frames = [train, validation, test, gap]
    categorical = ["Use Chip", "MCC", "Errors?"]
    if feature_mode == "clean":
        categorical.insert(1, "Merchant State")

    if use_amount_log:
        for frame in frames:
            frame["amount_log"] = np.log1p(np.maximum(frame["Amount"].to_numpy(), 0.0)).astype(np.float32)

    amount_scaler = StandardScaler().fit(train[["Amount"]])
    for frame in frames:
        frame["Amount"] = amount_scaler.transform(frame[["Amount"]]).astype(np.float32)
    log_scaler = None
    if use_amount_log:
        log_scaler = StandardScaler().fit(train[["amount_log"]])
        for frame in frames:
            frame["amount_log"] = log_scaler.transform(frame[["amount_log"]]).astype(np.float32)

    encoders: dict[str, dict[str, int]] = {}
    oov_counts: dict[str, dict[str, int]] = {}
    for column in categorical:
        for frame in frames:
            frame[column] = frame[column].fillna("<missing>").astype(str)
        mapping = {value: index + 1 for index, value in enumerate(sorted(train[column].unique()))}
        encoders[column] = mapping
        for frame in frames:
            frame[column] = frame[column].map(mapping).fillna(0).astype(np.int32)
        oov_counts[column] = {
            "validation": int((validation[column] == 0).sum()),
            "test": int((test[column] == 0).sum()),
            "gap": int((gap[column] == 0).sum()),
        }

    features = ["Amount"] + (["amount_log"] if use_amount_log else []) + categorical
    if use_cyclical_time:
        for frame in frames:
            hour = frame["datetime"].dt.hour + frame["datetime"].dt.minute / 60.0
            day = frame["datetime"].dt.dayofweek
            frame["hour_sin"] = np.sin(2 * np.pi * hour / 24).astype(np.float32)
            frame["hour_cos"] = np.cos(2 * np.pi * hour / 24).astype(np.float32)
            frame["dow_sin"] = np.sin(2 * np.pi * day / 7).astype(np.float32)
            frame["dow_cos"] = np.cos(2 * np.pi * day / 7).astype(np.float32)
        features += ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]

    forbidden = {"User", "Merchant Name", "Zip", "Merchant City", "Is Fraud?"}
    leaked = sorted(forbidden.intersection(features))
    if leaked:
        raise RuntimeError(f"Fitur terlarang masuk model: {leaked}")
    artifacts = {
        "features": features,
        "fit_scope": "train_only",
        "amount_scaler": {"mean": amount_scaler.mean_.tolist(), "scale": amount_scaler.scale_.tolist()},
        "amount_log_scaler": None if log_scaler is None else {"mean": log_scaler.mean_.tolist(), "scale": log_scaler.scale_.tolist()},
        "categorical_encoders": encoders,
        "oov_counts": oov_counts,
    }
    return train, validation, test, gap, features, artifacts


def smoke_subset(frame: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    """Small deterministic split that retains all positives up to one quarter of the limit."""
    if len(frame) <= max_rows:
        return frame.copy().reset_index(drop=True)
    positive = frame[frame["Is Fraud?"] == 1].iloc[: max_rows // 4]
    negative = frame[frame["Is Fraud?"] == 0].sample(max_rows - len(positive), random_state=seed)
    return pd.concat([positive, negative]).sort_values(["datetime", "transaction_id"]).reset_index(drop=True)



In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler

def graph_level_undersample(df: pd.DataFrame, ratio: float = 1.0, seed: int = 42) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp21d & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df
        
    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(
    df: pd.DataFrame,
    feature_cols: list,
    scalers_dict: dict = None,
    include_labels: bool = False,
) -> tuple:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi bersih (feature_cols). Label bersifat opsional.
      - user: Agregat rata-rata Amount & jumlah transaksi
      - merchant: Agregat rata-rata Amount & jumlah transaksi
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)

    Anti-Leakage Guard:
    Jika scalers_dict is None, scaler untuk User dan Merchant di-fit pada data saat ini (hanya untuk Train).
    Jika scalers_dict disediakan, scaler dari Train digunakan untuk me-transform User dan Merchant
    pada Validation dan Test tanpa membocorkan statistik masa depan.
    """
    print(f"Membangun HeteroData graf ({len(df)} transaksi)...")
    data = HeteroData()
    
    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # 2. Agregasi fitur User dan Merchant
    user_agg = df.groupby('User').agg(
        user_avg_amount=('Amount', 'mean'),
        user_tx_count=('Amount', 'count')
    ).reset_index()
    
    merchant_agg = df.groupby('Merchant Name').agg(
        merchant_avg_amount=('Amount', 'mean'),
        merchant_tx_count=('Amount', 'count')
    ).reset_index()
    
    # Pastikan urutan sesuai dengan mapping
    user_agg['mapped_id'] = user_agg['User'].map(user_mapping)
    user_agg = user_agg.sort_values('mapped_id').reset_index(drop=True)
    
    merchant_agg['mapped_id'] = merchant_agg['Merchant Name'].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values('mapped_id').reset_index(drop=True)
    
    # Penskalaan fitur User & Merchant (Strict Anti-Leakage)
    if scalers_dict is None:
        scaler_user = StandardScaler()
        user_feats_scaled = scaler_user.fit_transform(user_agg[['user_avg_amount', 'user_tx_count']])
        
        scaler_merchant = StandardScaler()
        merchant_feats_scaled = scaler_merchant.fit_transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        
        out_scalers = {
            'scaler_user': scaler_user,
            'scaler_merchant': scaler_merchant
        }
    else:
        scaler_user = scalers_dict['scaler_user']
        scaler_merchant = scalers_dict['scaler_merchant']
        user_feats_scaled = scaler_user.transform(user_agg[['user_avg_amount', 'user_tx_count']])
        merchant_feats_scaled = scaler_merchant.transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        out_scalers = scalers_dict
        
    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)
    
    # 3. Fitur transaksi. EXP 1C memakai include_labels=False agar encoder
    # random-init tidak mungkin mengakses target, bahkan secara tidak sengaja.
    tx_features = torch.tensor(df[feature_cols].values, dtype=torch.float)
    data['transaction'].x = tx_features
    if include_labels:
        data['transaction'].y = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    data['transaction'].num_nodes = num_transactions
    
    data['user'].x = user_features
    data['user'].num_nodes = num_users
    
    data['merchant'].x = merchant_features
    data['merchant'].num_nodes = num_merchants
    
    # 4. Membangun Edge Index Bipartit
    tx_indices = np.arange(num_transactions)
    user_src_indices = df['User'].map(user_mapping).values
    merchant_src_indices = df['Merchant Name'].map(merchant_mapping).values
    
    edge_user_to_tx = torch.tensor(np.array([user_src_indices, tx_indices]), dtype=torch.long)
    edge_tx_to_user = torch.tensor(np.array([tx_indices, user_src_indices]), dtype=torch.long)
    
    edge_merchant_to_tx = torch.tensor(np.array([merchant_src_indices, tx_indices]), dtype=torch.long)
    edge_tx_to_merchant = torch.tensor(np.array([tx_indices, merchant_src_indices]), dtype=torch.long)
    
    data['user', 'buys', 'transaction'].edge_index = edge_user_to_tx
    data['transaction', 'rev_buys', 'user'].edge_index = edge_tx_to_user
    
    data['merchant', 'sells', 'transaction'].edge_index = edge_merchant_to_tx
    data['transaction', 'rev_sells', 'merchant'].edge_index = edge_tx_to_merchant
    
    print(f"HeteroData selesai: {num_transactions} tx, {num_users} users, {num_merchants} merchants.")
    return data, out_scalers




In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self, 
        hidden_channels: int, 
        out_channels: int, 
        num_layers: int, 
        data_metadata: tuple, 
        feature_dims: dict, 
        aggr: str = 'mean',
        dropout: float = 0.2
    ):
        super().__init__()
        
        self.num_layers = num_layers
        self.aggr = aggr
        self.dropout_rate = dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr=aggr
                )
                for edge_type in data_metadata[1]
            })
            self.convs.append(conv)
            
            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier linear bawaan untuk monitoring val AUPRC selama pelatihan GNN
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict, return_embeddings=False):
        # 1. Proyeksi fitur awal
        h_dict = {}
        
        # Node yang memiliki fitur (transaction, user, merchant)
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
            elif node_type in self.param_embeddings:
                num_nodes = num_nodes_dict[node_type]
                h_dict[node_type] = self.param_embeddings[node_type].expand(num_nodes, -1)
                
        # 2. Message Passing Layer
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            if 'transaction' in h_dict and h_dict['transaction'].size(0) > 1:
                h_dict['transaction'] = self.bns[i](h_dict['transaction'])
            h_dict = {k: F.relu(v) for k, v in h_dict.items()}
            h_dict = {k: self.dropout(v) for k, v in h_dict.items()}
            
        tx_emb = h_dict['transaction']
        
        if return_embeddings:
            return tx_emb
            
        out = self.classifier(tx_emb)
        return out.squeeze(-1)

def manual_weighted_bce(logits, targets, pos_weight):
    """
    Weighted Binary Cross Entropy Loss stabil dengan koefisien pos_weight teredam.
    """
    max_val = torch.clamp(-logits, min=0)
    loss = (1 - targets) * logits + (1 + (pos_weight - 1) * targets) * (
        max_val + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()

def manual_unweighted_bce(logits, targets):
    """
    Standar Binary Cross Entropy Loss tanpa pembobotan untuk evaluasi objektif.
    """
    max_val = torch.clamp(-logits, min=0)
    loss = (1 - targets) * logits + (
        max_val + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()




In [ ]:
%%writefile samplers.py
from __future__ import annotations

from typing import Any

import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader


WEIGHT_ATTR = "sampling_weight"
USER_TO_TX = ("user", "buys", "transaction")
TX_TO_USER = ("transaction", "rev_buys", "user")
MERCHANT_TO_TX = ("merchant", "sells", "transaction")
TX_TO_MERCHANT = ("transaction", "rev_sells", "merchant")
WEIGHTED_RELATIONS = (TX_TO_USER, TX_TO_MERCHANT)


def counterpart_source(edge_type: tuple[str, str, str]) -> str:
    if edge_type == TX_TO_USER:
        return "merchant_degree"
    if edge_type == TX_TO_MERCHANT:
        return "user_degree"
    return "not_applicable"


def _bincount(index: torch.Tensor, size: int) -> torch.Tensor:
    return torch.bincount(index, minlength=size).to(dtype=torch.float32)


def _minmax_plus_floor(values: torch.Tensor, alpha: float) -> torch.Tensor:
    """Implement the predeclared min-max score + alpha formula."""
    if not 0.0 < alpha <= 1.0:
        raise ValueError(f"alpha harus berada pada (0, 1], diterima {alpha}")
    values = values.to(dtype=torch.float32)
    if values.numel() == 0:
        return values
    minimum, maximum = values.min(), values.max()
    if torch.isclose(maximum, minimum):
        return torch.ones_like(values)
    return (values - minimum) / (maximum - minimum + 1e-8) + alpha


def candidate_score(candidate_degree: torch.Tensor, method: str, alpha: float = 0.1) -> torch.Tensor:
    """Score competing transaction candidates from their counterpart degree."""
    method = method.lower()
    degree = candidate_degree.to(dtype=torch.float32).clamp_min(1.0)
    if method == "uniform":
        return torch.ones_like(degree)
    if method == "topology":
        raw = 1.0 / torch.log1p(degree)
    elif method == "importance":
        raw = torch.log1p(degree)
    else:
        raise ValueError(f"Metode sampling tidak didukung: {method}")
    return _minmax_plus_floor(raw, alpha)


def _transaction_counterpart_degrees(data: HeteroData) -> tuple[torch.Tensor, torch.Tensor]:
    required = {USER_TO_TX, TX_TO_USER, MERCHANT_TO_TX, TX_TO_MERCHANT}
    missing = required.difference(data.edge_types)
    if missing:
        raise ValueError(f"Relasi graf tripartit belum lengkap: {sorted(missing)}")

    transaction_store = data["transaction"]
    expected = int(transaction_store.num_nodes)
    if hasattr(transaction_store, "historical_user_degree") and hasattr(transaction_store, "historical_merchant_degree"):
        user_degree = transaction_store.historical_user_degree.to(torch.float32)
        merchant_degree = transaction_store.historical_merchant_degree.to(torch.float32)
        if user_degree.numel() != expected or merchant_degree.numel() != expected:
            raise ValueError("Shape historical counterpart degree tidak sama dengan jumlah transaksi")
        return user_degree, merchant_degree

    user_edges = data[USER_TO_TX].edge_index
    merchant_edges = data[MERCHANT_TO_TX].edge_index
    user_incoming = torch.bincount(user_edges[1], minlength=expected)
    merchant_incoming = torch.bincount(merchant_edges[1], minlength=expected)
    if not torch.equal(user_incoming, torch.ones_like(user_incoming)):
        raise ValueError("Setiap transaksi harus terhubung tepat ke satu user")
    if not torch.equal(merchant_incoming, torch.ones_like(merchant_incoming)):
        raise ValueError("Setiap transaksi harus terhubung tepat ke satu merchant")

    user_degree = _bincount(user_edges[0], int(data["user"].num_nodes))
    merchant_degree = _bincount(merchant_edges[0], int(data["merchant"].num_nodes))
    user_for_transaction = torch.empty(expected, dtype=torch.long)
    merchant_for_transaction = torch.empty(expected, dtype=torch.long)
    user_for_transaction[user_edges[1]] = user_edges[0]
    merchant_for_transaction[merchant_edges[1]] = merchant_edges[0]
    return user_degree[user_for_transaction], merchant_degree[merchant_for_transaction]


def _correlation(x: torch.Tensor, y: torch.Tensor) -> float | None:
    x, y = x.to(torch.float32), y.to(torch.float32)
    if x.numel() < 2 or torch.isclose(x.std(unbiased=False), torch.tensor(0.0)) or torch.isclose(y.std(unbiased=False), torch.tensor(0.0)):
        return None
    return float(torch.corrcoef(torch.stack([x, y]))[0, 1])


def apply_sampling_weights(
    data: HeteroData,
    method: str,
    alpha: float = 0.1,
    weight_attr: str = WEIGHT_ATTR,
) -> HeteroData:
    """
    Attach active edge weights for the second-hop transaction competition.

    Incoming edges sampled at a user destination use each candidate transaction's
    merchant degree. Incoming edges sampled at a merchant destination use each
    candidate transaction's user degree.
    """
    method = method.lower()
    user_degree_per_tx, merchant_degree_per_tx = _transaction_counterpart_degrees(data)
    for edge_type in data.edge_types:
        edge_index = data[edge_type].edge_index
        num_edges = int(edge_index.size(1))
        candidate_degree = torch.ones(num_edges, dtype=torch.float32)
        if edge_type == TX_TO_USER:
            candidate_degree = merchant_degree_per_tx[edge_index[0]]
        elif edge_type == TX_TO_MERCHANT:
            candidate_degree = user_degree_per_tx[edge_index[0]]
        weights = candidate_score(candidate_degree, method, alpha) if edge_type in WEIGHTED_RELATIONS else torch.ones(num_edges, dtype=torch.float32)
        if weights.numel() != num_edges or not torch.isfinite(weights).all() or (weights <= 0).any():
            raise ValueError(f"Bobot sampling invalid untuk relasi {edge_type}")
        data[edge_type][weight_attr] = weights.contiguous()
        data[edge_type].candidate_degree = candidate_degree.contiguous()
    validate_sampling_weight_contract(data, method, weight_attr)
    return data


def validate_sampling_weight_contract(data: HeteroData, method: str, weight_attr: str = WEIGHT_ATTR) -> None:
    method = method.lower()
    for edge_type in data.edge_types:
        store = data[edge_type]
        if weight_attr not in store:
            raise ValueError(f"{weight_attr} tidak ditemukan pada {edge_type}")
        weights = store[weight_attr]
        destination = store.edge_index[1]
        sums = torch.zeros(int(data[edge_type[2]].num_nodes), dtype=torch.float32)
        sums.scatter_add_(0, destination, weights)
        active = torch.bincount(destination, minlength=sums.numel()) > 0
        if active.any() and not (sums[active] > 0).all():
            raise ValueError(f"Jumlah bobot lokal non-positif pada {edge_type}")
        if edge_type not in WEIGHTED_RELATIONS and not torch.allclose(weights, torch.ones_like(weights)):
            raise ValueError(f"Relasi first-hop/non-kompetitif harus uniform: {edge_type}")
        if edge_type in WEIGHTED_RELATIONS:
            degree = store.candidate_degree.clamp_min(1.0)
            varies = torch.unique(degree).numel() > 1
            if method == "uniform" and not torch.allclose(weights, torch.ones_like(weights)):
                raise ValueError(f"Uniform sampler harus berbobot satu pada {edge_type}")
            if method != "uniform" and varies and torch.unique(weights).numel() <= 1:
                raise ValueError(f"Bobot {method} konstan meski counterpart degree bervariasi pada {edge_type}")
            correlation = _correlation(degree, weights)
            if method == "topology" and correlation is not None and correlation >= 0:
                raise ValueError(f"Topology weight harus menurun terhadap counterpart degree pada {edge_type}")
            if method == "importance" and correlation is not None and correlation <= 0:
                raise ValueError(f"Importance weight harus meningkat terhadap counterpart degree pada {edge_type}")


def sampling_weight_audit(data: HeteroData, method: str, weight_attr: str = WEIGHT_ATTR) -> dict[str, Any]:
    report: dict[str, Any] = {
        "method": method,
        "weight_attr": weight_attr,
        "weighted_relations": ["__".join(item) for item in WEIGHTED_RELATIONS],
        "relations": {},
    }
    for edge_type in data.edge_types:
        store = data[edge_type]
        weights = store[weight_attr].detach().cpu()
        degrees = store.candidate_degree.detach().cpu()
        report["relations"]["__".join(edge_type)] = {
            "num_edges": int(weights.numel()),
            "counterpart_source": counterpart_source(edge_type),
            "weight_min": float(weights.min()) if weights.numel() else None,
            "weight_max": float(weights.max()) if weights.numel() else None,
            "weight_mean": float(weights.mean()) if weights.numel() else None,
            "weight_std": float(weights.std(unbiased=False)) if weights.numel() else None,
            "unique_weights": int(torch.unique(weights).numel()),
            "candidate_degree_min": float(degrees.min()) if degrees.numel() else None,
            "candidate_degree_max": float(degrees.max()) if degrees.numel() else None,
            "degree_weight_correlation": _correlation(degrees.clamp_min(1.0), weights),
            "contract_passed": True,
        }
    return report


def get_neighbor_loader(
    data: HeteroData,
    method: str = "uniform",
    batch_size: int = 1024,
    shuffle: bool = False,
    num_neighbors: list[int] | None = None,
    seed: int = 42,
    alpha: float = 0.1,
    input_nodes: torch.Tensor | None = None,
    weight_attr: str = WEIGHT_ATTR,
) -> NeighborLoader:
    if num_neighbors is None:
        num_neighbors = [15, 10]
    if len(num_neighbors) != 2 or any(value <= 0 for value in num_neighbors):
        raise ValueError("num_neighbors harus berisi dua fan-out positif")
    apply_sampling_weights(data, method, alpha, weight_attr)
    if input_nodes is None:
        input_nodes = torch.arange(int(data["transaction"].num_nodes))
    generator = torch.Generator().manual_seed(seed)
    loader = NeighborLoader(
        data,
        num_neighbors={edge_type: list(num_neighbors) for edge_type in data.edge_types},
        batch_size=batch_size,
        input_nodes=("transaction", input_nodes),
        shuffle=shuffle,
        num_workers=0,
        weight_attr=weight_attr,
        generator=generator,
        subgraph_type="directional",
    )
    return loader



In [ ]:
%%writefile metrics.py
from __future__ import annotations

from typing import Any

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)


def find_optimal_threshold(probabilities: np.ndarray, labels: np.ndarray, min_precision: float = 0.15) -> tuple[float, dict[str, Any]]:
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    if thresholds.size == 0:
        return 0.5, {"tau": 0.5, "val_f1": 0.0, "candidate_thresholds": 0}
    p, r = precision[:-1], recall[:-1]
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    eligible = p >= min_precision
    indices = np.flatnonzero(eligible) if eligible.any() else np.arange(len(thresholds))
    best = int(indices[np.argmax(f1[indices])])
    return float(thresholds[best]), {
        "tau": float(thresholds[best]),
        "val_f1": float(f1[best]),
        "val_precision": float(p[best]),
        "val_recall": float(r[best]),
        "candidate_thresholds": int(len(thresholds)),
        "min_precision_constraint": float(min_precision),
        "constraint_applied": bool(eligible.any()),
    }


def evaluate_predictions(probabilities: np.ndarray, labels: np.ndarray, threshold: float) -> dict[str, Any]:
    predictions = (probabilities >= threshold).astype(np.int8)
    default = (probabilities >= 0.5).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    recall = float(recall_score(labels, predictions, zero_division=0))
    precision = float(precision_score(labels, predictions, zero_division=0))
    specificity = float(tn / (tn + fp)) if tn + fp else 0.0
    total_positive = int(labels.sum())
    operational: dict[str, Any] = {}
    for requested_k in (100, 200, 300, 421, 500, 1000):
        k = min(requested_k, len(labels))
        top = np.argpartition(probabilities, len(probabilities) - k)[-k:]
        captured = int(labels[top].sum())
        operational[str(requested_k)] = {
            "actual_k": int(k),
            "true_positives": captured,
            "recall": float(captured / total_positive) if total_positive else 0.0,
            "precision": float(captured / k) if k else 0.0,
        }
    return {
        "metrics_optimal_threshold": {
            "threshold": float(threshold),
            "f1_score": float(f1_score(labels, predictions, zero_division=0)),
            "recall": recall,
            "precision": precision,
            "accuracy": float(accuracy_score(labels, predictions)),
            "gmean": float(np.sqrt(recall * specificity)),
            "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        },
        "metrics_default_threshold_05": {
            "threshold": 0.5,
            "f1_score": float(f1_score(labels, default, zero_division=0)),
            "recall": float(recall_score(labels, default, zero_division=0)),
            "precision": float(precision_score(labels, default, zero_division=0)),
        },
        "metrics_threshold_independent": {
            "auprc": float(average_precision_score(labels, probabilities)),
            "roc_auc": float(roc_auc_score(labels, probabilities)),
            "recall_at_k": operational,
        },
    }


def threshold_curve(probabilities: np.ndarray, labels: np.ndarray, points: int = 181) -> dict[str, list[float]]:
    thresholds = np.linspace(0.05, 0.95, points)
    f1_values, recall_values, precision_values = [], [], []
    for threshold in thresholds:
        prediction = probabilities >= threshold
        f1_values.append(float(f1_score(labels, prediction, zero_division=0)))
        recall_values.append(float(recall_score(labels, prediction, zero_division=0)))
        precision_values.append(float(precision_score(labels, prediction, zero_division=0)))
    return {
        "threshold": thresholds.tolist(),
        "f1": f1_values,
        "recall": recall_values,
        "precision": precision_values,
    }



In [ ]:
%%writefile validate_frozen_control.py
from __future__ import annotations

import ast
import json
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
import torch

from graph_builder import build_hetero_graph
from model import HeteroGraphSAGE
from reproducibility import seed_everything
import samplers as sampler_module
from samplers import MERCHANT_TO_TX, TX_TO_MERCHANT, TX_TO_USER, USER_TO_TX, apply_sampling_weights, get_neighbor_loader, sampling_weight_audit


def toy_frame(labels: list[int]) -> pd.DataFrame:
    return pd.DataFrame({
        "User": [1, 1, 1, 2, 2, 3],
        "Merchant Name": [10, 10, 11, 10, 12, 12],
        "Amount": [1.0, 2.0, 3.0, 1.5, 4.0, 2.5],
        "f0": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6],
        "f1": [1.0, 0.0, 1.0, 0.0, 1.0, 0.0],
        "Is Fraud?": labels,
    })


def make_model(graph, seed: int) -> HeteroGraphSAGE:
    seed_everything(seed)
    model = HeteroGraphSAGE(128, 1, 2, graph.metadata(), {"transaction": 2, "user": 2, "merchant": 2}, "mean", 0.2)
    model.eval()
    counts = {node_type: graph[node_type].num_nodes for node_type in graph.node_types}
    with torch.no_grad():
        model(graph.x_dict, graph.edge_index_dict, counts, return_embeddings=True)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def embedding(model, graph) -> torch.Tensor:
    counts = {node_type: graph[node_type].num_nodes for node_type in graph.node_types}
    with torch.no_grad():
        return model(graph.x_dict, graph.edge_index_dict, counts, return_embeddings=True)


def main() -> None:
    root = Path(__file__).resolve().parent
    graph_a, scalers = build_hetero_graph(toy_frame([0, 1, 0, 0, 1, 0]), ["f0", "f1"], include_labels=False)
    graph_b, _ = build_hetero_graph(toy_frame([1, 0, 1, 1, 0, 1]), ["f0", "f1"], scalers, include_labels=False)
    model_a = make_model(graph_a, 42)
    model_b = make_model(graph_b, 42)
    embedding_a, embedding_b = embedding(model_a, graph_a), embedding(model_b, graph_b)
    source = (root / "main.py").read_text(encoding="utf-8")
    ast.parse(source)
    audits = {}
    for method in ("uniform", "topology", "importance"):
        candidate, _ = build_hetero_graph(toy_frame([0, 1, 0, 0, 1, 0]), ["f0", "f1"], scalers, include_labels=False)
        apply_sampling_weights(candidate, method)
        audits[method] = sampling_weight_audit(candidate, method)
    real_neighbor_loader = sampler_module.NeighborLoader

    class CaptureNeighborLoader:
        def __init__(self, *args, **kwargs):
            self.node_sampler = SimpleNamespace(weight_attr=kwargs.get("weight_attr"))

    sampler_module.NeighborLoader = CaptureNeighborLoader
    try:
        loader = get_neighbor_loader(graph_a, "importance", batch_size=2, num_neighbors=[2, 2], seed=42)
    finally:
        sampler_module.NeighborLoader = real_neighbor_loader
    duplicate, _ = build_hetero_graph(toy_frame([0, 1, 0, 0, 1, 0]), ["f0", "f1"], scalers, include_labels=False)
    duplicate[USER_TO_TX].edge_index = torch.cat([duplicate[USER_TO_TX].edge_index, duplicate[USER_TO_TX].edge_index[:, :1]], dim=1)
    duplicate_rejected = False
    try:
        apply_sampling_weights(duplicate, "topology")
    except ValueError:
        duplicate_rejected = True
    topology_relations = audits["topology"]["relations"]
    checks = {
        "graph_excludes_labels": "y" not in graph_a["transaction"] and "y" not in graph_b["transaction"],
        "embedding_shape_is_6x128": tuple(embedding_a.shape) == (6, 128),
        "all_encoder_parameters_frozen": all(not item.requires_grad for item in model_a.parameters()),
        "same_seed_same_state": all(torch.equal(model_a.state_dict()[key], model_b.state_dict()[key]) for key in model_a.state_dict()),
        "external_label_flip_does_not_change_embedding": torch.equal(embedding_a, embedding_b),
        "no_backward_in_main": ".backward(" not in source,
        "no_torch_optimizer_in_main": "torch.optim" not in source,
        "main_syntax_valid": True,
        "all_sampler_audits_pass": all(
            relation["contract_passed"]
            for audit in audits.values()
            for relation in audit["relations"].values()
        ),
        "tx_to_user_uses_merchant_degree": topology_relations["__".join(TX_TO_USER)]["counterpart_source"] == "merchant_degree",
        "tx_to_merchant_uses_user_degree": topology_relations["__".join(TX_TO_MERCHANT)]["counterpart_source"] == "user_degree",
        "first_hop_is_uniform": all(
            audits["topology"]["relations"]["__".join(edge)]["unique_weights"] == 1
            for edge in (USER_TO_TX, MERCHANT_TO_TX)
        ),
        "neighbor_loader_activates_weight_attr": loader.node_sampler.weight_attr == "sampling_weight",
        "duplicate_counterpart_is_rejected": duplicate_rejected,
    }
    output = {"status": "passed" if all(checks.values()) else "failed", "checks": checks, "sampler_audits": audits}
    (root / "validation_report.json").write_text(json.dumps(output, indent=2) + "\n", encoding="utf-8")
    for name, passed in checks.items():
        print(f"[{'PASS' if passed else 'FAIL'}] {name}")
    if not all(checks.values()):
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
import gc
import hashlib
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import torch
import yaml
from xgboost import XGBClassifier

from artifacts import RunLogger, build_artifact_index, write_json
from config import load_config
from data_preprocessing import apply_strict_anti_leakage_preprocessing, load_and_split_raw_data, smoke_subset
from graph_builder import build_hetero_graph
from metrics import evaluate_predictions, find_optimal_threshold, threshold_curve
from model import HeteroGraphSAGE
from reproducibility import environment_manifest, file_fingerprint, git_revision, seed_everything, snapshot_source, stable_hash
from samplers import apply_sampling_weights, get_neighbor_loader, sampling_weight_audit


MODEL_SPECS = {
    "tabular_10d": "10-d Clean Tabular + XGBoost",
    "frozen_graphsage_128d": "128-d Frozen Random GraphSAGE + XGBoost",
    "hybrid_tabular_frozen_138d": "10-d Tabular + 128-d Frozen GraphSAGE + XGBoost",
}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


def resolve_path(value: str, config_path: Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else (config_path.parent / path).resolve()


def resolve_dataset_path(value: str, config_path: Path) -> Path:
    """Resolve the configured dataset and tolerate Kaggle's alternate mount layout."""
    configured = resolve_path(value, config_path)
    if configured.is_file():
        return configured
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        matches = sorted(kaggle_input.rglob(configured.name))
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(
                f"Dataset {configured.name} ambigu di /kaggle/input: "
                + ", ".join(str(path) for path in matches)
            )
    raise FileNotFoundError(f"Dataset tidak ditemukan: {configured}")


def xgb_parameters(config: dict, class_ratio: float, seed: int, smoke: bool) -> dict[str, Any]:
    strategy = config.get("scale_pos_weight_strategy", "sqrt")
    scale = np.sqrt(class_ratio) if strategy == "sqrt" else class_ratio if strategy == "full" else 1.0
    return {
        "n_estimators": int(min(config["n_estimators"], 20) if smoke else config["n_estimators"]),
        "max_depth": int(config["max_depth"]),
        "learning_rate": float(config["learning_rate"]),
        "min_child_weight": float(config["min_child_weight"]),
        "subsample": float(config["subsample"]),
        "colsample_bytree": float(config["colsample_bytree"]),
        "scale_pos_weight": float(scale),
        "tree_method": config.get("tree_method", "hist"),
        "random_state": seed,
        "n_jobs": int(config.get("n_jobs", -1)),
        "eval_metric": "logloss",
    }


def materialize_model(model: HeteroGraphSAGE, loader, device: torch.device) -> None:
    model.eval()
    with torch.no_grad():
        batch = next(iter(loader)).to(device)
        counts = {node_type: batch[node_type].num_nodes for node_type in batch.node_types}
        model(batch.x_dict, batch.edge_index_dict, counts, return_embeddings=True)


def prepare_frozen_encoder(
    graph,
    loader,
    cfg: dict,
    device: torch.device,
    checkpoint_path: Path,
    encoder_seed: int,
) -> tuple[HeteroGraphSAGE, dict[str, Any]]:
    seed_everything(encoder_seed, cfg["reproducibility"]["deterministic"], cfg["reproducibility"]["warn_only"])
    model_cfg = cfg["model"]
    model = HeteroGraphSAGE(
        hidden_channels=int(model_cfg["hidden_channels"]),
        out_channels=int(model_cfg["out_channels"]),
        num_layers=int(model_cfg["num_layers"]),
        data_metadata=graph.metadata(),
        feature_dims={"transaction": int(cfg["features"]["input_dim_expected"]), "user": 2, "merchant": 2},
        aggr=model_cfg["aggr"],
        dropout=float(model_cfg["dropout"]),
    ).to(device)
    materialize_model(model, loader, device)
    expected = {
        "encoder_seed": encoder_seed,
        "hidden_channels": int(model_cfg["hidden_channels"]),
        "num_layers": int(model_cfg["num_layers"]),
        "num_neighbors": list(cfg["training"]["num_neighbors"]),
        "feature_dims": {"transaction": int(cfg["features"]["input_dim_expected"]), "user": 2, "merchant": 2},
    }
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    if checkpoint_path.exists():
        payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
        if payload.get("contract") != expected:
            raise RuntimeError("Checkpoint encoder beku ada, tetapi kontraknya berbeda")
        model.load_state_dict(payload["state_dict"])
        checkpoint_action = "reused"
    else:
        torch.save({"contract": expected, "state_dict": model.state_dict()}, checkpoint_path)
        checkpoint_action = "created"
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    if trainable != 0 or model.training:
        raise RuntimeError("Kontrak encoder beku gagal")
    provenance = {
        **expected,
        "checkpoint": str(checkpoint_path.resolve()),
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "checkpoint_action": checkpoint_action,
        "training_steps": 0,
        "optimizer_created": False,
        "labels_in_graph": "y" in graph["transaction"],
        "trainable_parameters": trainable,
        "model_mode": "eval",
    }
    if provenance["labels_in_graph"]:
        raise RuntimeError("Label ditemukan di graf encoder")
    return model, provenance


def extract_frozen_embeddings(model, loader, device: torch.device, total: int, width: int, output_path: Path) -> float:
    matrix = np.lib.format.open_memmap(output_path, mode="w+", dtype=np.float32, shape=(total, width))
    offset = 0
    started = time.time()
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            counts = {node_type: batch[node_type].num_nodes for node_type in batch.node_types}
            embedding = model(batch.x_dict, batch.edge_index_dict, counts, return_embeddings=True)
            batch_size = int(batch["transaction"].batch_size)
            matrix[offset : offset + batch_size] = embedding[:batch_size].cpu().numpy()
            offset += batch_size
    matrix.flush()
    del matrix
    if offset != total:
        raise RuntimeError(f"Ekstraksi embedding tidak lengkap: {offset}/{total}")
    return time.time() - started


def create_hybrid_memmap(tabular: np.ndarray, embedding_path: Path, output_path: Path) -> None:
    embedding = np.load(embedding_path, mmap_mode="r")
    width = tabular.shape[1] + embedding.shape[1]
    hybrid = np.lib.format.open_memmap(output_path, mode="w+", dtype=np.float32, shape=(len(tabular), width))
    block = 100_000
    for start in range(0, len(tabular), block):
        stop = min(start + block, len(tabular))
        hybrid[start:stop, : tabular.shape[1]] = tabular[start:stop]
        hybrid[start:stop, tabular.shape[1] :] = embedding[start:stop]
    hybrid.flush()
    del hybrid, embedding


def train_downstream(
    name: str,
    train_x,
    validation_x,
    test_x,
    labels: dict[str, np.ndarray],
    cfg: dict,
    seed: int,
    smoke: bool,
    run_dir: Path,
) -> tuple[dict[str, Any], np.ndarray, np.ndarray]:
    positives = int(labels["train"].sum())
    class_ratio = float((len(labels["train"]) - positives) / max(positives, 1))
    parameters = xgb_parameters(cfg["classifier"], class_ratio, seed, smoke)
    classifier = XGBClassifier(**parameters)
    started = time.time()
    classifier.fit(train_x, labels["train"])
    validation_probability = classifier.predict_proba(validation_x)[:, 1]
    test_probability = classifier.predict_proba(test_x)[:, 1]
    threshold, selection = find_optimal_threshold(
        validation_probability, labels["validation"], float(cfg["threshold"]["min_precision"])
    )
    result = {
        "display_name": MODEL_SPECS[name],
        "dimensions": int(train_x.shape[1]),
        "xgboost_parameters": parameters,
        "threshold_selection": selection,
        "validation": evaluate_predictions(validation_probability, labels["validation"], threshold),
        "test": evaluate_predictions(test_probability, labels["test"], threshold),
        "threshold_curve_validation": threshold_curve(validation_probability, labels["validation"]),
        "fit_and_predict_seconds": time.time() - started,
    }
    classifier.save_model(run_dir / f"xgboost_{name}.json")
    return result, validation_probability.astype(np.float32), test_probability.astype(np.float32)


def run_experiment(args: argparse.Namespace) -> Path:
    config_path = Path(args.config).resolve()
    cfg = load_config(str(config_path))
    method = args.sampling.lower()
    if method not in cfg["sampling"]["methods"]:
        raise ValueError(f"Sampler {method} tidak didaftarkan di konfigurasi")
    seed = int(args.seed if args.seed is not None else cfg["experiment"]["seed"])
    encoder_seed = int(cfg["experiment"]["encoder_seed"])
    evaluation_seed = int(cfg["experiment"]["evaluation_seed"])
    seed_everything(seed, cfg["reproducibility"]["deterministic"], cfg["reproducibility"]["warn_only"])
    device = torch.device(args.device or ("cuda" if torch.cuda.is_available() else "cpu"))
    source_dir = Path(__file__).resolve().parent
    result_root = resolve_path(cfg["output"]["result_dir"], config_path)
    model_root = resolve_path(cfg["output"]["model_dir"], config_path)
    config_hash = stable_hash({"config": cfg, "sampling": method, "smoke": args.smoke})
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = args.run_id or f"exp21d_v6_1c_{method}_seed{seed}_{config_hash}_{stamp}"
    run_dir = result_root / ("smoke" if args.smoke else "runs") / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    args._active_run_dir = run_dir
    logger = RunLogger(run_dir / "events.jsonl")
    manifest = {
        "experiment_id": cfg["experiment"]["id"],
        "control": "frozen_random_init_graphsage",
        "run_id": run_id,
        "status": "running",
        "sampling": method,
        "seed": seed,
        "encoder_seed": encoder_seed,
        "device": str(device),
        "config_hash": config_hash,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "git": git_revision(source_dir),
    }
    write_json(run_dir / "manifest.json", manifest)
    logger.log("run_started", **{key: manifest[key] for key in ("run_id", "sampling", "device")})
    if cfg["output"].get("snapshot_source", True):
        snapshot_source(source_dir, run_dir / "code_snapshot")
    (run_dir / "resolved_config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    write_json(run_dir / "environment.json", environment_manifest())

    dataset_path = resolve_dataset_path(cfg["data"]["transactions"], config_path)
    manifest["dataset"] = file_fingerprint(dataset_path, cfg["reproducibility"]["compute_dataset_sha256"])
    write_json(run_dir / "manifest.json", manifest)
    time_cfg = cfg["time_window"]
    train, validation, test, gap, statistics = load_and_split_raw_data(
        str(dataset_path), cfg["data"]["start_year"], cfg["data"].get("max_rows"),
        time_cfg["reference_date"], time_cfg["train_days"], time_cfg["val_days"],
        time_cfg["gap_days"], time_cfg["test_days"],
    )
    if args.smoke:
        limit = int(cfg["smoke"]["max_rows_per_split"])
        train, validation, test = (
            smoke_subset(train, limit, seed), smoke_subset(validation, limit, seed + 1), smoke_subset(test, limit, seed + 2)
        )
        gap = smoke_subset(gap, min(limit, len(gap)), seed + 3)
    feature_cfg = cfg["features"]
    train, validation, test, gap, features, preprocessing = apply_strict_anti_leakage_preprocessing(
        train, validation, test, gap, feature_cfg["mode"], feature_cfg["use_cyclical_time"], feature_cfg["use_amount_log"]
    )
    if len(features) != int(feature_cfg["input_dim_expected"]):
        raise RuntimeError(f"Kontrak fitur gagal: {len(features)} != {feature_cfg['input_dim_expected']}")
    write_json(run_dir / "preprocessing_artifacts.json", preprocessing)
    write_json(run_dir / "window_statistics.json", statistics)
    labels = {name: frame["Is Fraud?"].to_numpy(dtype=np.int8) for name, frame in (("train", train), ("validation", validation), ("test", test))}
    ids = {name: frame["transaction_id"].to_numpy(dtype=np.int64) for name, frame in (("train", train), ("validation", validation), ("test", test))}
    tabular = {name: np.ascontiguousarray(frame[features].to_numpy(dtype=np.float32)) for name, frame in (("train", train), ("validation", validation), ("test", test))}

    train_graph, scalers = build_hetero_graph(train, features, include_labels=False)
    validation_graph, _ = build_hetero_graph(validation, features, scalers, include_labels=False)
    test_graph, _ = build_hetero_graph(test, features, scalers, include_labels=False)
    graphs = {"train": train_graph, "validation": validation_graph, "test": test_graph}
    if any("y" in graph["transaction"] for graph in graphs.values()):
        raise RuntimeError("Label tidak boleh berada di graf EXP 1C")
    del train, validation, test, gap
    gc.collect()

    sampling_cfg = cfg["sampling"]
    audits = {}
    for split, graph in graphs.items():
        apply_sampling_weights(graph, method, float(sampling_cfg["smoothing_alpha"]), sampling_cfg["weight_attr"])
        audits[split] = sampling_weight_audit(graph, method, sampling_cfg["weight_attr"])
    write_json(run_dir / "sampler_weight_audit.json", {"status": "passed", "method": method, "graphs": audits})
    loaders = {
        split: get_neighbor_loader(
            graph, method, int(cfg["training"]["batch_size"]), False, cfg["training"]["num_neighbors"],
            evaluation_seed + index, float(sampling_cfg["smoothing_alpha"]), weight_attr=sampling_cfg["weight_attr"],
        )
        for index, (split, graph) in enumerate(graphs.items())
    }
    checkpoint = model_root / f"frozen_random_encoder_seed{encoder_seed}.pt"
    model, provenance = prepare_frozen_encoder(train_graph, loaders["train"], cfg, device, checkpoint, encoder_seed)
    write_json(run_dir / "encoder_provenance.json", provenance)
    write_json(run_dir / "encoder_history.json", {"epochs": 0, "training_steps": 0, "loss_history": [], "optimizer": None})
    logger.log("frozen_encoder_ready", checkpoint_sha256=provenance["checkpoint_sha256"], action=provenance["checkpoint_action"])

    feature_dir = run_dir / "features"
    feature_dir.mkdir()
    inference_seconds = {}
    for split in ("train", "validation", "test"):
        np.save(feature_dir / f"{split}_tabular_10d.npy", tabular[split])
        np.save(feature_dir / f"{split}_labels.npy", labels[split])
        np.save(feature_dir / f"{split}_transaction_ids.npy", ids[split])
        inference_seconds[split] = extract_frozen_embeddings(
            model, loaders[split], device, len(labels[split]), int(cfg["model"]["hidden_channels"]),
            feature_dir / f"{split}_frozen_graphsage_128d.npy",
        )
        create_hybrid_memmap(
            tabular[split], feature_dir / f"{split}_frozen_graphsage_128d.npy",
            feature_dir / f"_{split}_hybrid_working_138d.npy",
        )
    contract = {
        "row_alignment": "input_nodes arange, NeighborLoader shuffle=false; every .npy row aligns by split",
        "representations": MODEL_SPECS,
        "feature_names": features,
        "embedding_dtype": "float32",
        "hybrid_reconstruction": "concatenate tabular_10d and frozen_graphsage_128d column-wise; temporary 138-d matrices are deleted after XGBoost fitting",
        "labels_excluded_from_encoder_graph": True,
        "checkpoint_sha256": provenance["checkpoint_sha256"],
    }
    write_json(feature_dir / "feature_contract.json", contract)
    logger.log("features_saved", inference_seconds=inference_seconds)

    matrices = {}
    for split in ("train", "validation", "test"):
        embedding = np.load(feature_dir / f"{split}_frozen_graphsage_128d.npy", mmap_mode="r")
        hybrid = np.load(feature_dir / f"_{split}_hybrid_working_138d.npy", mmap_mode="r")
        matrices[split] = {
            "tabular_10d": tabular[split],
            "frozen_graphsage_128d": embedding,
            "hybrid_tabular_frozen_138d": hybrid,
        }
    results, prediction_payload = {}, {}
    for name in MODEL_SPECS:
        result, validation_probability, test_probability = train_downstream(
            name, matrices["train"][name], matrices["validation"][name], matrices["test"][name],
            labels, cfg, seed, args.smoke, run_dir,
        )
        results[name] = result
        prediction_payload[f"validation_probability__{name}"] = validation_probability
        prediction_payload[f"test_probability__{name}"] = test_probability
        logger.log("downstream_model_complete", model=name, test=result["test"])
    del matrices, embedding, hybrid
    gc.collect()
    for split in ("train", "validation", "test"):
        (feature_dir / f"_{split}_hybrid_working_138d.npy").unlink()
    np.savez_compressed(
        run_dir / "predictions.npz",
        validation_labels=labels["validation"], test_labels=labels["test"],
        validation_transaction_ids=ids["validation"], test_transaction_ids=ids["test"],
        **prediction_payload,
    )
    summary = {
        "experiment_id": cfg["experiment"]["id"], "sampling": method, "seed": seed,
        "scientific_control": "random_init_graphsage_frozen_zero_training_steps",
        "encoder_provenance": provenance, "inference_seconds": inference_seconds, "models": results,
    }
    write_json(run_dir / "metrics.json", summary)
    manifest.update({"status": "complete", "completed_at_utc": datetime.now(timezone.utc).isoformat(), "encoder_checkpoint_sha256": provenance["checkpoint_sha256"]})
    write_json(run_dir / "manifest.json", manifest)
    logger.log("run_complete", artifact_count=build_artifact_index(run_dir)["file_count"])
    return run_dir


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="EXP 1C: frozen random-init GraphSAGE control")
    parser.add_argument("--config", default="config.reference.yaml")
    parser.add_argument("--sampling", choices=["uniform", "topology", "importance"], required=True)
    parser.add_argument("--seed", type=int)
    parser.add_argument("--device")
    parser.add_argument("--run-id")
    parser.add_argument("--smoke", action="store_true")
    return parser.parse_args()


if __name__ == "__main__":
    arguments = parse_args()
    try:
        output = run_experiment(arguments)
        print(f"Run complete: {output}")
    except Exception as error:
        active = getattr(arguments, "_active_run_dir", None)
        if active:
            write_json(Path(active) / "failure.json", {"error": repr(error), "traceback": traceback.format_exc()})
            manifest_path = Path(active) / "manifest.json"
            if manifest_path.exists():
                import json
                manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
                manifest["status"] = "failed"
                write_json(manifest_path, manifest)
        raise


In [ ]:
%%writefile visualize_results.py
from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import precision_recall_curve


COLORS = {"uniform": "#4472C4", "topology": "#ED7D31", "importance": "#70AD47"}
LABELS = {
    "tabular_10d": "Tabular 10-d",
    "frozen_graphsage_128d": "Frozen GNN 128-d",
    "hybrid_tabular_frozen_138d": "Hybrid 138-d",
}


def save_figure(fig, output: Path, stem: str) -> list[str]:
    paths = []
    for suffix in ("png", "pdf"):
        path = output / f"{stem}.{suffix}"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        paths.append(path.name)
    plt.close(fig)
    return paths


def load_runs(paths: list[str]) -> list[dict]:
    runs = []
    for value in paths:
        root = Path(value).resolve()
        metrics = json.loads((root / "metrics.json").read_text(encoding="utf-8"))
        predictions = np.load(root / "predictions.npz")
        runs.append({"root": root, "metrics": metrics, "predictions": predictions})
    return runs


def main() -> None:
    parser = argparse.ArgumentParser(description="Publication-ready figures for EXP 1C")
    parser.add_argument("--runs", nargs="+", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()
    output = Path(args.output).resolve()
    output.mkdir(parents=True, exist_ok=True)
    runs = load_runs(args.runs)
    generated, source_rows = [], []

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    models = list(LABELS)
    x = np.arange(len(models))
    width = 0.8 / len(runs)
    for index, run in enumerate(runs):
        method = run["metrics"]["sampling"]
        auprc, f1 = [], []
        for name in models:
            result = run["metrics"]["models"][name]["test"]
            auprc.append(result["metrics_threshold_independent"]["auprc"])
            f1.append(result["metrics_optimal_threshold"]["f1_score"])
            source_rows.append({"figure": "performance", "sampling": method, "model": name, "metric": "auprc", "value": auprc[-1]})
            source_rows.append({"figure": "performance", "sampling": method, "model": name, "metric": "f1", "value": f1[-1]})
        offset = (index - (len(runs) - 1) / 2) * width
        axes[0].bar(x + offset, auprc, width, label=method.title(), color=COLORS.get(method))
        axes[1].bar(x + offset, f1, width, label=method.title(), color=COLORS.get(method))
    for axis, title, ylabel in ((axes[0], "Test AUPRC", "AUPRC"), (axes[1], "Test F1 at validation-selected threshold", "F1")):
        axis.set_title(title)
        axis.set_ylabel(ylabel)
        axis.set_xticks(x, [LABELS[item] for item in models], rotation=15, ha="right")
        axis.grid(axis="y", alpha=0.25)
    axes[0].legend(frameon=False)
    fig.suptitle("EXP 1C — Frozen random-init GraphSAGE controls")
    generated += save_figure(fig, output, "fig1_frozen_control_performance")

    fig, axis = plt.subplots(figsize=(7.5, 5.5))
    for run in runs:
        method = run["metrics"]["sampling"]
        labels = run["predictions"]["test_labels"]
        for name, linestyle in (("frozen_graphsage_128d", "--"), ("hybrid_tabular_frozen_138d", "-")):
            probability = run["predictions"][f"test_probability__{name}"]
            precision, recall, _ = precision_recall_curve(labels, probability)
            axis.plot(recall, precision, linestyle=linestyle, color=COLORS.get(method), label=f"{method.title()} — {LABELS[name]}")
            stride = max(1, len(precision) // 1000)
            for p, r in zip(precision[::stride], recall[::stride]):
                source_rows.append({"figure": "pr_curve", "sampling": method, "model": name, "metric": "precision_at_recall", "value": float(p), "recall": float(r)})
    baseline = float(np.mean(runs[0]["predictions"]["test_labels"]))
    axis.axhline(baseline, color="black", linewidth=1, alpha=0.5, label=f"Prevalence = {baseline:.4f}")
    axis.set(xlabel="Recall", ylabel="Precision", title="Test precision–recall curves")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8, frameon=False, ncol=2)
    generated += save_figure(fig, output, "fig2_precision_recall_curves")

    fig, axes = plt.subplots(1, len(runs), figsize=(5 * len(runs), 4.2), squeeze=False)
    for axis, run in zip(axes[0], runs):
        method = run["metrics"]["sampling"]
        cm = run["metrics"]["models"]["hybrid_tabular_frozen_138d"]["test"]["metrics_optimal_threshold"]["confusion_matrix"]
        matrix = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
        image = axis.imshow(matrix, cmap="Blues")
        for row in range(2):
            for column in range(2):
                axis.text(column, row, f"{matrix[row, column]:,}", ha="center", va="center")
        axis.set(title=method.title(), xlabel="Predicted", ylabel="Actual", xticks=[0, 1], yticks=[0, 1])
        fig.colorbar(image, ax=axis, fraction=0.046)
    fig.suptitle("Hybrid frozen-control confusion matrices")
    generated += save_figure(fig, output, "fig3_hybrid_confusion_matrices")

    fig, axes = plt.subplots(1, len(runs), figsize=(5 * len(runs), 4.5), squeeze=False)
    rng = np.random.default_rng(42)
    for axis, run in zip(axes[0], runs):
        method = run["metrics"]["sampling"]
        embedding = np.load(run["root"] / "features" / "test_frozen_graphsage_128d.npy", mmap_mode="r")
        labels = np.load(run["root"] / "features" / "test_labels.npy", mmap_mode="r")
        positive = np.flatnonzero(labels == 1)
        negative = np.flatnonzero(labels == 0)
        pos_take = rng.choice(positive, min(len(positive), 2500), replace=False) if len(positive) else np.array([], dtype=int)
        neg_take = rng.choice(negative, min(len(negative), 2500), replace=False) if len(negative) else np.array([], dtype=int)
        chosen = np.concatenate([pos_take, neg_take])
        coordinates = PCA(n_components=2, random_state=42).fit_transform(np.asarray(embedding[chosen]))
        chosen_labels = np.asarray(labels[chosen])
        axis.scatter(coordinates[chosen_labels == 0, 0], coordinates[chosen_labels == 0, 1], s=5, alpha=0.25, label="Non-fraud")
        axis.scatter(coordinates[chosen_labels == 1, 0], coordinates[chosen_labels == 1, 1], s=8, alpha=0.55, label="Fraud")
        axis.set(title=method.title(), xlabel="PC1", ylabel="PC2")
        axis.legend(frameon=False, markerscale=2)
        with (output / f"pca_source_{method}.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle)
            writer.writerow(["transaction_row", "label", "pc1", "pc2"])
            writer.writerows(zip(chosen.tolist(), chosen_labels.tolist(), coordinates[:, 0].tolist(), coordinates[:, 1].tolist()))
    fig.suptitle("PCA of frozen random GraphSAGE test embeddings")
    generated += save_figure(fig, output, "fig4_frozen_embedding_pca")

    fieldnames = sorted({key for row in source_rows for key in row})
    with (output / "figure_source_data.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(source_rows)
    manifest = {"status": "complete", "figures": generated, "source_data": ["figure_source_data.csv", *[f"pca_source_{run['metrics']['sampling']}.csv" for run in runs]]}
    (output / "figures_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(output)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_all.py
from __future__ import annotations

import argparse
import csv
import json
import subprocess
import sys
from pathlib import Path

from config import load_config


def latest_run(result_root: Path, method: str, seed: int, smoke: bool) -> Path:
    parent = result_root / ("smoke" if smoke else "runs")
    candidates = sorted(parent.glob(f"exp21d_v6_1c_{method}_seed{seed}_*"), key=lambda item: item.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(f"Run {method} tidak ditemukan di {parent}")
    return candidates[-1]


def main() -> None:
    parser = argparse.ArgumentParser(description="Run all frozen GraphSAGE controls sequentially")
    parser.add_argument("--config", default="config.reference.yaml")
    parser.add_argument("--seed", type=int)
    parser.add_argument("--device")
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    source = Path(__file__).resolve().parent
    config_path = Path(args.config).resolve()
    cfg = load_config(str(config_path))
    seed = int(args.seed if args.seed is not None else cfg["experiment"]["seed"])
    result_root = Path(cfg["output"]["result_dir"])
    if not result_root.is_absolute():
        result_root = (config_path.parent / result_root).resolve()

    run_dirs = []
    for method in cfg["sampling"]["methods"]:
        command = [sys.executable, str(source / "main.py"), "--config", str(config_path), "--sampling", method, "--seed", str(seed)]
        if args.device:
            command += ["--device", args.device]
        if args.smoke:
            command.append("--smoke")
        subprocess.run(command, cwd=source, check=True)
        run_dirs.append(latest_run(result_root, method, seed, args.smoke))

    report_dir = result_root / ("smoke_summary" if args.smoke else "summary") / f"seed_{seed}"
    report_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    hashes = set()
    for run_dir in run_dirs:
        payload = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
        hashes.add(payload["encoder_provenance"]["checkpoint_sha256"])
        for name, model in payload["models"].items():
            optimal = model["test"]["metrics_optimal_threshold"]
            independent = model["test"]["metrics_threshold_independent"]
            rows.append({
                "sampling": payload["sampling"], "model": name, "dimensions": model["dimensions"],
                "auprc": independent["auprc"], "roc_auc": independent["roc_auc"],
                "f1": optimal["f1_score"], "precision": optimal["precision"], "recall": optimal["recall"],
                "threshold": optimal["threshold"], "run_dir": str(run_dir.resolve()),
            })
    if len(hashes) != 1:
        raise RuntimeError(f"Sampler tidak memakai checkpoint encoder identik: {sorted(hashes)}")
    with (report_dir / "metrics_summary.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    (report_dir / "run_directories.json").write_text(
        json.dumps({"encoder_checkpoint_sha256": next(iter(hashes)), "runs": [str(item.resolve()) for item in run_dirs]}, indent=2) + "\n",
        encoding="utf-8",
    )
    subprocess.run(
        [sys.executable, str(source / "visualize_results.py"), "--output", str(report_dir), "--runs", *map(str, run_dirs)],
        cwd=source, check=True,
    )
    print(f"All controls complete: {report_dir}")


if __name__ == "__main__":
    main()


In [ ]:
import subprocess
import sys
import torch
torch_version = torch.__version__.split('+')[0]
cuda = torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+cu{cuda}.html' if cuda != 'cpu' else f'https://data.pyg.org/whl/torch-{torch_version}+cpu.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric', 'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost', 'scikit-learn', 'matplotlib', 'PyYAML'], check=True)


In [ ]:
!python validate_frozen_control.py


In [ ]:
!python run_all.py --config config.kaggle.yaml --seed 42
